In [1]:
import sys
print("Python executable:", sys.executable)
print("Module search paths:", sys.path)

Python executable: /Users/peterbertels/credit_card_fraud_detection/venv/bin/python
Module search paths: ['/Library/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/peterbertels/credit_card_fraud_detection/venv/lib/python3.11/site-packages']


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
from imblearn.over_sampling import SMOTE  # For handling imbalance

# Set plot style
plt.style.use('seaborn-v0_8')

# Load data
df = pd.read_csv("../data/creditcard.csv")
print("Data loaded. Shape:", df.shape)

# Sample a larger subset for better representation
df = df.sample(n=50000, random_state=42)
print("Sampled Data Shape:", df.shape)

# Basic info
print("\nDataset Info:")
print(df.info())
print("\nClass Distribution:")
print(df["Class"].value_counts())

# Visualize class imbalance
plt.figure(figsize=(6, 4))
sns.countplot(x="Class", data=df)
plt.title("Class Distribution (0: Non-Fraud, 1: Fraud)")
plt.savefig("class_distribution.png")
plt.close()

# Visualize feature distributions (example: V1-V3 features)
plt.figure(figsize=(10, 6))
for i in range(1, 4):
    plt.subplot(3, 1, i)
    sns.histplot(df[f"V{i}"], bins=50, kde=True)
    plt.title(f"Distribution of V{i}")
plt.tight_layout()
plt.savefig("feature_distributions.png")
plt.close()

# Preprocess data
X = df.drop("Class", axis=1).values
y = df["Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print("\nAfter SMOTE - X_train shape:", X_train.shape, "y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape, "y_test shape:", y_test.shape)

# Train Random Forest model
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=50, max_depth=10, min_samples_split=2, random_state=42)
rf.fit(X_train, y_train)
print("Training completed.")

# Evaluate model
y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

# Print full metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig("confusion_matrix.png")
plt.close()

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc_score(y_test, y_pred_proba):.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.savefig("roc_curve.png")
plt.close()

# Feature importance
feature_importance = pd.DataFrame({'feature': df.drop('Class', axis=1).columns, 'importance': rf.feature_importances_})
feature_importance = feature_importance.sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance)
plt.title("Feature Importance")
plt.savefig("feature_importance.png")
plt.close()

print("\nVisualizations saved: class_distribution.png, feature_distributions.png, confusion_matrix.png, roc_curve.png, feature_importance.png")

Data loaded. Shape: (284807, 31)
Sampled Data Shape: (50000, 31)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 50000 entries, 43428 to 274591
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    50000 non-null  float64
 1   V1      50000 non-null  float64
 2   V2      50000 non-null  float64
 3   V3      50000 non-null  float64
 4   V4      50000 non-null  float64
 5   V5      50000 non-null  float64
 6   V6      50000 non-null  float64
 7   V7      50000 non-null  float64
 8   V8      50000 non-null  float64
 9   V9      50000 non-null  float64
 10  V10     50000 non-null  float64
 11  V11     50000 non-null  float64
 12  V12     50000 non-null  float64
 13  V13     50000 non-null  float64
 14  V14     50000 non-null  float64
 15  V15     50000 non-null  float64
 16  V16     50000 non-null  float64
 17  V17     50000 non-null  float64
 18  V18     50000 non-null  float64
 19  V19     50000 non-null  float